# AI Agent Sandbox with Search, Calculator & arXiv Research Tools
This notebook demonstrates a working AI Agent using LangChain, Google Gemini, Google Serper Search, Calculator, and arXiv paper search.

In [15]:
# Optional: install dependencies if not already installed
# !pip install langchain langchain-core langchain-community pydantic langchain-google-genai requests ipykernel ddgs google-generativeai python-dotenv

In [16]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

True

In [17]:
from langchain_community.utilities import GoogleSerperAPIWrapper

# Test Serper Search wrapper
search = GoogleSerperAPIWrapper()
result = search.run("Top 10 news of India Today")
print("Search Result:", result)

Search Result: Rahul Gandhi targets Amit Shah over police action on students, government hits back. Lok Sabha witnesses chaos as Treasury benches object to LoP's use of ... India News. Only teargas was fired: Govt hits back at Rahul Gandhi over student shooting claim · Global. Cheap Thailand trip turns into horror, 3 Indians ... Pleas in Supreme Court allege illegal detentions by Delhi and Bihar police · Aaratrika Bhaumik · Cockroach Janta Party supporters celebrate as Union Education ... Amit Shah ordered use of force on students, claims Rahul; BJP asks for proof · Rahul Gandhi to hold press conf at 6 pm; Anti-paper leak bill passed in LS · India, ... India's top court orders no coercive action against youth protesters and release of minors, media reports say India's Supreme Court on Tuesday Get all the latest news, live updates and content about India from across the BBC. Live Asia China India India's 'cockroach' protest called off after education ... 'Learn Some Skill': Kangana Rana

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the working Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.3,
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [19]:
# Test Gemini model invocation
response = llm.invoke("Hello! Introduce yourself briefly.")
raw_content = response.content
clean_reply = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in raw_content) if isinstance(raw_content, list) else str(raw_content)
print("Gemini Response:", clean_reply)

c:\Users\hp\OneDrive\Desktop\AI AGENT\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Gemini Response: Hello! I'm an AI assistant created by Google. I'm here to help you answer questions, solve problems, write text, brainstorm ideas, and learn new things. How can I help you today?


In [20]:
from langchain_core.tools import tool
import math

# Define a Calculator tool
@tool
def calculator(expression: str) -> str:
    """Calculates the result of a mathematical expression.
    Input should be a valid mathematical expression string (e.g. '2 + 2', '15 * 8', '100 / 4', '2**10', 'sqrt(144)').
    """
    try:
        allowed_names = {
            'abs': abs, 'round': round, 'pow': pow, 'min': min, 'max': max,
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
            'log': math.log, 'pi': math.pi, 'e': math.e
        }
        result = eval(expression, {'__builtins__': None}, allowed_names)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

# Test calculator tool standalone
print("Calculator test output:", calculator.invoke("15 * 8 + 42"))

Calculator test output: 162


In [21]:
from langchain_core.tools import tool
import arxiv

# Define arXiv Research Search Tool
@tool
def arxiv_search(query: str) -> str:
    """Searches scientific research papers on arXiv.
    Input should be a search query topic or research domain (e.g. 'quantum computing', 'transformer models', 'artificial intelligence').
    Returns paper titles, authors, summaries, and PDF links for top relevant research papers.
    """
    try:
        client = arxiv.Client()
        search = arxiv.Search(query=query, max_results=3, sort_by=arxiv.SortCriterion.Relevance)
        results = list(client.results(search))
        if not results:
            return 'No research papers found for the query.'
        output = []
        for i, paper in enumerate(results, 1):
            authors = ', '.join(a.name for a in paper.authors[:3])
            output.append(
                f"""Paper {i}:
Title: {paper.title}
Authors: {authors}
Published: {paper.published.strftime('%Y-%m-%d')}
Summary: {paper.summary[:300]}...
PDF Link: {paper.pdf_url}"""
            )
        return '\n\n'.join(output)
    except Exception as e:
        return f"Error searching arXiv: {e}"

# Test arXiv search tool standalone
print("arXiv Search Test:", arxiv_search.invoke("quantum computing"))

SyntaxError: unterminated string literal (detected at line 21) (2641719073.py, line 21)

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# Create agent equipped with Search, Calculator, and arXiv Research tools
agent = create_agent(
    model=llm,
    tools=[search.run, calculator, arxiv_search],
    system_prompt=(
        "You are a helpful AI research assistant equipped with Google Search, a Calculator, and an arXiv Research tool. "
        "Use Google Search for real-time web news and prices, Calculator for math calculations, "
        "and arXiv Search when users ask for scientific or academic research papers."
    ),
    checkpointer=MemorySaver()
)

In [ ]:
# Test query asking for research papers on arXiv
question = 'Find top 2 recent research papers on quantum computing on arXiv and summarize them.'

In [ ]:
# Corrected input key from 'message' to 'messages' (plural)
response = agent.invoke({"messages": [{"role": "user", "content": question}]},{"configurable":{"thread_id":"Prathamesh"}})
raw_reply = response["messages"][-1].content
clean_reply = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in raw_reply) if isinstance(raw_reply, list) else str(raw_reply)
print("Response:", clean_reply)